# Kaza — Standalone Preflight Check (A100)

Independently verifies NoPE hooks (`6/6`), sustained HF streaming speed vs. GPU speed, codec frame count, and peak 60 s (`750`-frame) VRAM including AdamW state before starting `phase1_train.ipynb`.

In [ ]:
import os
import subprocess
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    print("Add HF_TOKEN in Colab Secrets, grant this notebook access, then rerun this cell.")

REPO = "/content/kaza"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "origin", "phase1-design"], check=True)
else:
    subprocess.run(["git", "clone", "-b", "phase1-design", "https://github.com/udohchuks/kaza.git", REPO], check=True)
%cd /content/kaza
!pip install -q -r requirements.txt

In [ ]:
from kaza.env import print_env
from kaza.codec import KazaCodec
from kaza.data import WaxalStreamingDataset, discover_waxal_sources
from kaza.train import TrainConfig

env = print_env()
cfg = TrainConfig(frame_lengths=(250, 375, 750), batch_size=4, grad_accum=8, prefetch_batches=4)
codec = KazaCodec(device=env.device, dtype=env.frozen_dtype)

train_sources, train_weights, _ = discover_waxal_sources("google/WaxalNLP", include_tts=True, lang_alpha=0.5)
train_stream = WaxalStreamingDataset(train_sources, source_weights=train_weights, clip_frames=max(cfg.frame_lengths))

In [ ]:
from kaza.model import KazaModel
from kaza.preflight import preflight_check

model = KazaModel(
    device=env.device,
    frozen_dtype=env.frozen_dtype,
    trainable_dtype=env.trainable_dtype,
    autocast_dtype=env.autocast_dtype,
    drop_prob=0.08,
    mtp_offsets=(2, 4),
    mtp_weight=0.2,
    nope_new_layers=True,
)
preflight_check(model, codec, train_stream, cfg)